# Wahkon Demo — Profile-Objective Deep RKHS Superposition Network

This notebook demonstrates:
1. Installing Wahkon from GitHub
2. Creating a synthetic regression dataset
3. Selecting regularization hyperparameters via Bayesian optimization
4. Training a ProfileWKN model
5. Point prediction and RMSE evaluation

**Target function** (f₃ from the paper):
$$f_3(x) = \exp\!\Bigl(\tfrac{1}{2}\bigl[\sin\bigl(\pi(x_1^2+x_2^2)\bigr) + \sin\bigl(\pi(x_3^2+x_4^2)\bigr)\bigr]\Bigr) + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 0.42^2)$$

## 1. Installation

In [ ]:
# Install Wahkon from GitHub (change URL to your repo)
!pip install -q git+https://github.com/YOUR_USERNAME/wahkon.git

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from wahkon import ProfileWKN, create_dataset, select_lambda_twostage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Define the Target Function and Create Dataset

In [ ]:
# f₃ (D=4): exp(0.5 * [sin(π(x₁² + x₂²)) + sin(π(x₃² + x₄²))]) + noise
NOISE_STD = 0.420433

f_noisy = lambda x: (
    torch.exp(0.5 * (
        torch.sin(torch.pi * (x[:, [0]] ** 2 + x[:, [1]] ** 2))
        + torch.sin(torch.pi * (x[:, [2]] ** 2 + x[:, [3]] ** 2))))
    + NOISE_STD * torch.randn(x.shape[0], 1)
)
f_true = lambda x: torch.exp(0.5 * (
    torch.sin(torch.pi * (x[:, [0]] ** 2 + x[:, [1]] ** 2))
    + torch.sin(torch.pi * (x[:, [2]] ** 2 + x[:, [3]] ** 2))))

# Settings
N_VAR   = 4
N_TRAIN = 400
N_TEST  = 1000
SEED    = 42
WIDTH   = [4, 4, 2, 1]    # 3-layer WKN matching the additive structure
GRID    = 9
SIGMA   = 0.5
STEPS   = 500
LR      = 0.005

dataset = create_dataset(
    f_noisy, f_true=f_true, n_var=N_VAR,
    ranges=(-1, 1), train_num=N_TRAIN, test_num=N_TEST,
    normalize_input=True, normalize_label=True,
    seed=SEED, device=device,
)

print(f'Train: {dataset["train_input"].shape}')
print(f'Test:  {dataset["test_input"].shape}')

## 3. Hyperparameter Selection (Fixed λ_lower + BO for λ_last)

Wahkon uses two independent regularization parameters:
- **`lamb_lower`**: RKHS penalty for lower-layer link functions. **Fixed** by the deterministic formula: $\lambda_{\mathrm{lower}} = n^{-4/5} \times \#\text{links}$, where $\#\text{links} = \sum_l D_l \times D_{l+1}$.
- **`lamb_last`**: Last-layer profile regularization. Selected via **1D Bayesian optimisation** over K-fold cross-validation RMSE.

In [ ]:
def num_link_fun(width):
    return sum(width[l] * width[l + 1] for l in range(len(width) - 1))

scale = N_TRAIN ** (-4/5) * num_link_fun(WIDTH)
fixed_lamb_lower = scale

print(f'Lambda scale factor: {scale:.6f}')
print(f'lamb_lower (fixed) = {fixed_lamb_lower:.6f}')

In [ ]:
# Lambda selection: lamb_lower is FIXED, lamb_last selected via 1D BO over CV-RMSE
# (This takes a few minutes — set SKIP_BO=True to use defaults instead)
SKIP_BO = False

if SKIP_BO:
    best_lamb_lower = fixed_lamb_lower
    best_lamb_last  = scale * 0.5
    print('Using default lambdas (skipped BO)')
else:
    print('Running 1D BO for lamb_last (lamb_lower is fixed)...')
    best_lamb_last, best_lamb_lower = select_lambda_twostage(
        width=WIDTH, dataset=dataset,
        n_splits=5, steps=STEPS, lr=LR,
        grid=GRID, sigma=SIGMA,
        n_calls=15, n_random_init=5,
        lamb_last_range=(0.01, 3.0),
        batch=200,
        device=device, random_state=SEED, seed=SEED,
    )

print(f'lamb_lower = {best_lamb_lower:.6f}  (fixed)')
print(f'lamb_last  = {best_lamb_last:.6f}  (BO-selected)')

## 4. Train ProfileWKN

In [ ]:
model = ProfileWKN(
    width=WIDTH, grid=GRID, sigma=SIGMA,
    seed=SEED, device=device,
)

results, _, _ = model.train(
    dataset, opt='Adam', steps=STEPS, lr=LR,
    lamb_last=best_lamb_last, lamb_lower=best_lamb_lower,
    batch=200, update_grid=False,
    verbose=True, device=device,
    early_stopping=True, patience=50, min_delta=1e-5,
)

# Refit last layer on full training set
model.fit_last_layer(
    dataset['train_input'], dataset['train_label'],
    lamb_last=best_lamb_last, device=device,
)

## 5. Evaluate

In [ ]:
with torch.no_grad():
    f_hat = model.predict(
        dataset['train_input'], dataset['test_input'], device=device,
    )
    y_true = dataset['test_true'].view(-1).to(device)
    test_rmse = float(torch.sqrt(torch.mean((y_true - f_hat) ** 2)))

print(f'Test RMSE (vs true function): {test_rmse:.4f}')

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: RMSE curves
axes[0].plot(results['train_loss'], label='Train RMSE', alpha=0.8)
axes[0].plot(results['test_loss'], label='Test RMSE', alpha=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Training Curves')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)

# Panel 2: Profile objective
axes[1].plot(results['profile_obj'], color='tab:red', alpha=0.8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Profile Objective')
axes[1].set_title('Profile Objective')
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'f₃ (D=4)  |  width={WIDTH}  |  n={N_TRAIN}  |  RMSE={test_rmse:.4f}',
    fontsize=13, y=1.02,
)
plt.tight_layout()
plt.show()

## 7. Prediction vs Ground Truth

Since f₃ is 4-dimensional, we visualize 2D slices by fixing $(x_3, x_4) = (0, 0)$ and varying $(x_1, x_2)$, then fixing $(x_1, x_2) = (0, 0)$ and varying $(x_3, x_4)$.

In [ ]:
n_grid = 80
g = torch.linspace(-1, 1, n_grid)
ga, gb = torch.meshgrid(g, g, indexing='ij')

# Training normalization statistics
train_mean = dataset['train_input'].mean(dim=0, keepdim=True)
train_std  = dataset['train_input'].std(dim=0, keepdim=True).clamp(min=1e-6)
label_mean = dataset['train_label'].cpu().mean()
label_std  = dataset['train_label'].cpu().std().clamp(min=1e-6)

def make_slice_and_predict(vary_dims, fix_val=0.0):
    """Create a 2D grid varying `vary_dims`, fixing the other two at `fix_val`."""
    x_grid = torch.full((n_grid * n_grid, 4), fix_val)
    x_grid[:, vary_dims[0]] = ga.flatten()
    x_grid[:, vary_dims[1]] = gb.flatten()
    x_grid = x_grid.to(device)

    # True function
    y_true_grid = f_true(x_grid).cpu().reshape(n_grid, n_grid)
    y_true_norm = (y_true_grid - label_mean) / label_std

    # WKN prediction (normalize inputs)
    x_grid_norm = (x_grid - train_mean) / train_std
    with torch.no_grad():
        y_pred = model.predict(
            dataset['train_input'], x_grid_norm, device=device,
        ).cpu().reshape(n_grid, n_grid)

    return y_true_norm, y_pred

# Slice 1: vary (x₁, x₂), fix (x₃, x₄) = (0, 0)
y_true_12, y_pred_12 = make_slice_and_predict([0, 1])

# Slice 2: vary (x₃, x₄), fix (x₁, x₂) = (0, 0)
y_true_34, y_pred_34 = make_slice_and_predict([2, 3])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, (y_true_s, y_pred_s, vary_label) in enumerate([
    (y_true_12, y_pred_12, '(x₁, x₂),  x₃=x₄=0'),
    (y_true_34, y_pred_34, '(x₃, x₄),  x₁=x₂=0'),
]):
    vmin = min(y_true_s.min(), y_pred_s.min())
    vmax = max(y_true_s.max(), y_pred_s.max())
    diff = (y_pred_s - y_true_s).numpy()

    im0 = axes[row, 0].contourf(ga.numpy(), gb.numpy(), y_true_s.numpy(),
                                levels=30, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f'True f₃  |  vary {vary_label}', fontsize=11)
    plt.colorbar(im0, ax=axes[row, 0])

    im1 = axes[row, 1].contourf(ga.numpy(), gb.numpy(), y_pred_s.numpy(),
                                levels=30, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[row, 1].set_title(f'WKN Prediction  |  vary {vary_label}', fontsize=11)
    plt.colorbar(im1, ax=axes[row, 1])

    im2 = axes[row, 2].contourf(ga.numpy(), gb.numpy(), diff,
                                levels=30, cmap='RdBu_r')
    axes[row, 2].set_title(f'Error  |  vary {vary_label}', fontsize=11)
    plt.colorbar(im2, ax=axes[row, 2])

plt.suptitle(
    f'ProfileWKN  |  f₃ (D=4)  |  width={WIDTH}  |  n={N_TRAIN}  |  RMSE={test_rmse:.4f}',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()